- Read all bronze tables
- Clean and join into one wide table
- Run silver data quality checks
- Write to silver Delta table
- Log run to audit table

In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"])
dbutils.widgets.dropdown("run_mode", "full", ["full", "incremental"])

env = dbutils.widgets.get("environment")
print(env)
run_mode = dbutils.widgets.get("run_mode")
print(run_mode)

In [0]:
%run /Workspace/Users/yash.karda.yk@gmail.com/commerce-data-platform/utils/data_quality.py

In [0]:
%run /Workspace/Users/yash.karda.yk@gmail.com/commerce-data-platform/utils/helpers.py

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
       StructType,StructField,
       StringType,
       IntegerType,
       LongType,
       TimestampType,
       DoubleType
       )

configs = get_config(env)
print(configs)
base_path = configs["base_path"]
layers = configs["layers"]


In [0]:
## Defining Paths
RAW_PATH    = f"{base_path}/raw"
BRONZE_PATH = f"{base_path}/bronze"
SILVER_PATH = f"{base_path}/silver"
GOLD_PATH   = f"{base_path}/gold"
AUDIT_PATH  = f"{base_path}/audit"

print(f"RAW    → {RAW_PATH}")
print(f"BRONZE → {BRONZE_PATH}")
print(f"SILVER → {SILVER_PATH}")
print(f"GOLD   → {GOLD_PATH}")
print(f"AUDIT  → {AUDIT_PATH}")

In [0]:
from datetime import datetime
start_time = datetime.now()
total_rows = 0

df_orders = read_delta(spark, f"{BRONZE_PATH}/orders")
df_customers = read_delta(spark, f"{BRONZE_PATH}/customers")
df_products = read_delta(spark, f"{BRONZE_PATH}/products")
df_sellers = read_delta(spark, f"{BRONZE_PATH}/sellers")
df_payments = read_delta(spark, f"{BRONZE_PATH}/payments")
df_order_items = read_delta(spark, f"{BRONZE_PATH}/order_items")
df_reviews = read_delta(spark, f"{BRONZE_PATH}/reviews")
df_translation = read_delta(spark, f"{BRONZE_PATH}/translation")

print("All bronze tables loaded!")

In [0]:
# Drop audit columns
audit_cols = ["ingestion_date", "source_file_name"]
df_orders = df_orders.drop(*audit_cols)
df_customers = df_customers.drop(*audit_cols)
df_products = df_products.drop(*audit_cols)
df_sellers = df_sellers.drop(*audit_cols)
df_payments = df_payments.drop(*audit_cols)
df_order_items = df_order_items.drop(*audit_cols)
df_reviews = df_reviews.drop(*audit_cols)
df_translation = df_translation.drop(*audit_cols)

# Deduplicate and filter nulls
df_orders = df_orders.dropDuplicates().filter(F.col("order_id").isNotNull())
df_customers = df_customers.dropDuplicates().filter(F.col("customer_id").isNotNull())
df_products = df_products.dropDuplicates().filter(F.col("product_id").isNotNull())
df_sellers = df_sellers.dropDuplicates().filter(F.col("seller_id").isNotNull())
df_payments = df_payments.dropDuplicates().filter(F.col("order_id").isNotNull())
df_order_items = df_order_items.dropDuplicates().filter(F.col("order_id").isNotNull())
df_reviews = df_reviews.dropDuplicates().filter(F.col("order_id").isNotNull())
df_translation = df_translation.dropDuplicates().filter(F.col("product_category_name").isNotNull())

# Join all tables
from pyspark.sql.functions import broadcast

df_silver = df_orders \
    .join(df_customers, on="customer_id", how="left") \
    .join(df_order_items, on="order_id", how="left") \
    .join(df_products, on="product_id", how="left") \
    .join(df_sellers, on="seller_id", how="left") \
    .join(df_payments, on="order_id", how="left") \
    .join(df_reviews, on="order_id", how="left") \
    .join(broadcast(df_translation), on="product_category_name", how="left")

# Filter delivered orders only
df_silver = df_silver.filter(F.col("order_status") == "delivered")
df_silver = df_silver.filter(F.col("payment_value").isNotNull())

print(f"Silver table row count: {df_silver.count()}")

In [0]:
silver_config = layers["silver"]

check_row_count(df_silver, silver_config["table_name"])
check_no_nulls(df_silver, silver_config["table_name"], silver_config["critical_columns"])
check_value_range(df_silver, "silver", "payment_value", 0, 999999)

print("All silver checks passed!")

In [0]:
# Write silver table
write_delta(df_silver, f"{SILVER_PATH}/order_details", "overwrite")
total_rows = df_silver.count()
print(f"Silver table written: {total_rows} rows")

# Log run
end_time = datetime.now()
log_run(
    spark=spark,
    notebook_name="job_02_transform",
    environment=env,
    run_mode=run_mode,
    start_time=start_time,
    end_time=end_time,
    rows_processed=total_rows,
    status="SUCCESS",
    audit_path=AUDIT_PATH,
    error_message=None
)

print("Run logged successfully!")

dbutils.notebook.exit("SUCCESS")